# Copilot Medallion — orchestrator

Deployed by `copilot.roesli.org`. Writes a progress log to `Files/_run.log` in the target lakehouse so failures are diagnosable from OneLake.

In [ ]:
# default parameters (overridden by Job Scheduler)
source_lakehouse_id = ""
source_tables_csv = ""
target_lakehouse_id = ""
target_lakehouse_name = ""
workspace_id = ""
run_id = "local-dev"
spec_url = ""

In [ ]:
from datetime import datetime
import traceback

tables = [t.strip() for t in source_tables_csv.split(',') if t.strip()]
src_base = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{source_lakehouse_id}"
tgt_base = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{target_lakehouse_id}"
log_path = f"{tgt_base}/Files/_run.log"
log_lines = []
def log(msg):
    line = f"{datetime.utcnow().isoformat()}Z | {msg}"
    log_lines.append(line)
    print(line)
def flush_log():
    try:
        from pyspark.sql import Row
        rdd = spark.sparkContext.parallelize([Row(line=l) for l in log_lines])
        spark.createDataFrame(rdd).coalesce(1).write.mode('overwrite').option('header','false').text(log_path.replace('.log','.txt'))
    except Exception as e:
        print(f'flush_log failed: {e}')

log(f"run_id={run_id}")
log(f"src_base={src_base}")
log(f"tgt_base={tgt_base}")
log(f"tables={tables}")

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.utils import AnalysisException

bronze_results = {}
try:
    for t in tables:
        flat = t.replace('/', '_').lower()
        src_path = f"{src_base}/Tables/{t}"
        tgt_path = f"{tgt_base}/Tables/bronze_{flat}"
        log(f"bronze: reading {src_path}")
        try:
            df = spark.read.format('delta').load(src_path)
            df.write.format('delta').mode('overwrite').option('overwriteSchema','true').save(tgt_path)
            n = df.count()
            bronze_results[t] = n
            log(f"bronze_{flat}: rows={n}")
        except Exception as e:
            bronze_results[t] = f"ERROR: {e}"
            log(f"bronze_{flat}: ERROR {type(e).__name__}: {e}")
except Exception as e:
    log(f"FATAL bronze loop: {type(e).__name__}: {e}")
    log(traceback.format_exc())
    flush_log(); raise

In [ ]:
silver_results = {}
ts = F.lit(datetime.utcnow().isoformat())
try:
    for t in tables:
        if isinstance(bronze_results.get(t), str):
            continue
        flat = t.replace('/', '_').lower()
        bronze_path = f"{tgt_base}/Tables/bronze_{flat}"
        silver_path = f"{tgt_base}/Tables/silver_{flat}"
        df = spark.read.format('delta').load(bronze_path)
        total = df.count()
        null_counts = df.select([F.sum(F.col(c).isNull().cast('int')).alias(c) for c in df.columns]).collect()[0].asDict()
        keep = [c for c, n in null_counts.items() if n < total]
        df2 = df.select(*[F.trim(F.col(c)).alias(c) if str(df.schema[c].dataType).startswith('StringType') else F.col(c) for c in keep])
        df2 = df2.withColumn('_silver_ts', ts)
        df2.write.format('delta').mode('overwrite').option('overwriteSchema','true').save(silver_path)
        silver_results[t] = df2.count()
        log(f"silver_{flat}: rows={silver_results[t]}")
except Exception as e:
    log(f"FATAL silver loop: {type(e).__name__}: {e}")
    log(traceback.format_exc())
    flush_log(); raise

In [ ]:
gold_built = False
try:
    lc = {t.split('/')[-1].lower(): t for t in tables}
    awl_keys = {'customer','product','salesorderheader','salesorderdetail'}
    if awl_keys.issubset(set(lc.keys())):
        def silver(name):
            return spark.read.format('delta').load(f"{tgt_base}/Tables/silver_{lc[name].replace('/','_').lower()}")
        cust = silver('customer'); prod = silver('product')
        soh = silver('salesorderheader'); sod = silver('salesorderdetail')
        cust.write.format('delta').mode('overwrite').option('overwriteSchema','true').save(f"{tgt_base}/Tables/dim_customer")
        prod.write.format('delta').mode('overwrite').option('overwriteSchema','true').save(f"{tgt_base}/Tables/dim_product")
        dates = soh.select(F.to_date('OrderDate').alias('Date')).distinct().filter('Date is not null')
        dates = (dates.withColumn('Year', F.year('Date')).withColumn('Quarter', F.quarter('Date'))
                       .withColumn('Month', F.month('Date')).withColumn('Day', F.dayofmonth('Date')))
        dates.write.format('delta').mode('overwrite').option('overwriteSchema','true').save(f"{tgt_base}/Tables/dim_date")
        fact = (sod.alias('d').join(soh.alias('h'), F.col('d.SalesOrderID') == F.col('h.SalesOrderID'), 'inner')
                .select(F.col('h.CustomerID').alias('CustomerKey'), F.col('d.ProductID').alias('ProductKey'),
                        F.to_date('h.OrderDate').alias('DateKey'), F.col('d.OrderQty').alias('Qty'),
                        F.col('d.UnitPrice').alias('UnitPrice'), F.col('d.LineTotal').alias('LineTotal')))
        fact.write.format('delta').mode('overwrite').option('overwriteSchema','true').save(f"{tgt_base}/Tables/fact_sales")
        gold_built = True
        log(f"gold_built=True")
    else:
        log(f"gold skipped: tables {sorted(lc.keys())} don't match AdventureWorksLT shape")
except Exception as e:
    log(f"FATAL gold: {type(e).__name__}: {e}")
    log(traceback.format_exc())
    flush_log(); raise

In [ ]:
log(f"SUMMARY bronze={bronze_results} silver={silver_results} gold_built={gold_built}")
flush_log()
print('done')